#### MongoDB Implementation Work

In [ ]:
import json
import datetime
from pymongo import MongoClient
import pandas as pd

In [ ]:
db_name = 'fleximart'
collection = 'products-catelog'
mongo_url = 'mongodb://localhost:27017'

#### Operation 1: Load Data

In [ ]:
filename = 'products_catalog.json'

with open(filename, 'r') as file:
    products = json.load(file)

In [ ]:
client = MongoClient(mongo_url)
db = client[db_name]
collection = db[collection]

In [ ]:
collection.delete_many({})
rec = collection.insert_many(products)
print(f"inserted {len(rec.inserted_ids)} documents into {db_name}.{collection}")

#### Operation 2: Basic Query 

- Find all products in Electronics category with price less than 50000
- Return only: name, price and stock 

###### Here - I came up with two approaches. Doing it directly from the file and alternative is from the DB. 

Approach # 1 using the file 

In [ ]:
#Electronic_Products_Price_lessthan_5K = [
#    category for category in products
#    if category['category'] == 'Electronics' and category['price'] < 5000
#]

Products_Condition = [product for product in products 
                     if product["category"] == 'Electronics' and product['price'] < 50000] 
Filtered_Products = []
for product in Products_Condition:
    Filtered_Products = {field: product.get(field, None) for field in ['name', 'price', 'stock']} 

print(json.dumps(Filtered_Products, indent=4))

Approach # 2 using the DB data

In [ ]:
Filtered_Products_Using_DB_Search = collection.find(
    {"category": 'Electronics', "price": {"$lt": 50000}}, 
    {"_id": 0, "name": 1, "price": 1, "stock": 1},
)

pd.DataFrame(list(Filtered_Products_Using_DB_Search))


#### Operation 3: Review Analysis

- Find all products that have average rating >= 4.0 
- Use aggregation to calculate average from reviews array 

###### Here - I came up with two approaches. Doing it directly from the file and alternative is from the DB. 

Approach # 1 using the file 

In [ ]:
for item in products:
    scores = [review['rating'] for review in item['reviews']]
    if scores:
        average_score = sum(scores) / len(scores)
        item['average_score'] = round(average_score, 2)
    else:
        item['average_score'] = 0

    Rating_Condition = [rating for rating in products
            if rating['average_score'] >= 4.0 ]

print(json.dumps(Rating_Condition, indent=4))   

Approach # 2 using the DB 

In [ ]:
review_analysis = [
    { 
        "$addFields": {
            "avg_rating": {
                "$avg": {"$ifNull": ["$reviews.rating", []]}
            }
    }
    },
    {"$match": {"avg_rating": {"$gte": 4.0}}},
    {
        "$project": {
            "_id": 0,
            "product_id": 1,
            "name": 1,
            "category": 1,
            "avg_rating": {"$round": ["$avg_rating", 2]}
        }
    }
]

pd.DataFrame(list(collection.aggregate(review_analysis)))

#### Operation 4: Update Operation 

- Add a new review to product "ELEC001"
- Review: {user: "U999", rating: 4, comment: "Good value", data: ISODate()}

Approach # 1 - Using File and updating directly the file

In [ ]:
# 1. Modify the Python object
for item in products:
    if item['product_id'] == 'ELEC001':
        item["reviews"].append({"user": "U999", "rating": 4, "comment": "Good value", "date": "ISODate()"})
        print(item)
        break
    else:
        print(f"Error: Product ID {'ELEC001'} not found.")


Approach # 2 - Using DB and updating DB 

In [ ]:
add_review = {
    "user": "U999",
    "rating": 4,
    "comment": "Good_Value",
    "date": datetime.datetime.now(),
}

In [ ]:
resultAfterAddReview = collection.update_one(
    {"product_id": "ELEC001"},
    {"$push": {"reviews": add_review}},
)

print({"matched": resultAfterAddReview.matched_count, "modified": resultAfterAddReview.modified_count})

In [ ]:
updated = collection.find_one(
    {"product_id": "ELEC001"},
    {"_id": 0, "product_id": 1, "name": 1, "reviews": 1}
)

updated

Operation 5: Complex Aggregation

Approach # 1 using and updating JSON File

In [ ]:
from collections import defaultdict
# Aggregation containers
category_totals = defaultdict(lambda: {"total_price": 0, "count": 0})

# Aggregate data
for item in products:
    cat = item["category"]
    price = item["price"]

    category_totals[cat]["total_price"] += price
    category_totals[cat]["count"] += 1

# Build final result
result = []
for category, values in category_totals.items():
    avg_price = values["total_price"] / values["count"]
    result.append({
        "category": category,
        "avg_price": avg_price,
        "product_count": values["count"]
    })

# Sort by avg_price descending
result_sorted = sorted(result, key=lambda x: x["avg_price"], reverse=True)

# Print output
for row in result_sorted:
    print(row)


Appraoch 2 - Using DB 

In [52]:
complexAggregation = [
    {
        "$group": {
            "_id": "$category",
            "avg_price": {"$avg": "$price"},
            "product_count": {"$sum": 1},
        }
    },
    {
        "$project": {
            "_id": 0,
            "category": "$_id",
            "avg_price": {"$round": ["$avg_price", 2]},
            "product_count": 1,
        }
    },
    {"$sort": {"avg_price": -1}},
]

pd.DataFrame(list(collection.aggregate(complexAggregation)))

,product_count,category,avg_price
0,6,Electronics,70830.83
1,6,Fashion,5215.00
